In [1]:
import numpy as np
import torch
from scipy.integrate import solve_ivp

# ---------------------------
# Lorenz system definition
# ---------------------------
# Lorenz 96 (monoscale)
def lorenz96(t, x, F=8.0):
    N = len(x)
    dxdt = np.zeros(N)
    for i in range(N):
        dxdt[i] = (x[(i + 1) % N] - x[i - 2]) * x[i - 1] - x[i] + F
    return dxdt

# Parameters
F = 8.0
N = 100
dt = 0.001
maxtime = 5000
t_eval = np.linspace(0, maxtime, int(maxtime / dt))

# Initial condition
np.random.seed(2025)
y0 = F * np.ones(N) + 0.1 * np.random.randn(N)

# Solve ODE
sol = solve_ivp(lambda t, x: lorenz96(t, x, F), (0, maxtime), y0, t_eval=t_eval, method='RK45')


In [2]:
sol.y.shape # Rule of thumb: 5–10× more points than features to avoid overfitting 
# let k = 2 and d =100 then nonlinear feature vector size formula dk(dk+1)/2. 
#  200*201/2 is 20100
# let k = 10 and d =100 then 1000*1001/2 is 500500

(100, 5000000)

In [3]:
# Subsample → shape (N, n_points)
data_clean = sol.y[:, ::10].astype(np.float32)
print(data_clean.shape, data_clean.dtype)
data_clean.shape

# ---------------------------
# Data Splits
# ---------------------------
T = data_clean.shape[1]      # total time points
warmup_len = 1000            # fixed warmup segment

# Compute split sizes
train_len = int(0.8 * T) - warmup_len
val_len   = int(0.1 * T)
test_len  = int(0.1 * T)

print(f"Total length: {T}")
print(f"Warmup: {warmup_len}, Train: {train_len}, Val: {val_len}, Test: {test_len}")


(100, 500000) float32
Total length: 500000
Warmup: 1000, Train: 399000, Val: 50000, Test: 50000


In [4]:
# ---------------------------
# Noise utility (float32 safe)
# ---------------------------
def add_relative_gaussian_noise(X_clean, noise_scale):
    """
    Adds relative Gaussian noise to dataset X_clean.
    Returns float32.
    """
    X_clean = X_clean.astype(np.float32)  # ensure float32 input
    signal_std = np.std(X_clean, axis=1, keepdims=True)
    noise = np.random.normal(0.0, noise_scale * signal_std, X_clean.shape).astype(np.float32)
    return X_clean + noise

# ---------------------------
# Generate and Save Multiple Noise Levels (float32 safe)
# ---------------------------

np.random.seed(2026)
noise_levels = [0.01, 0.1, 0.2, 0.3]

for nl in noise_levels:
    # Add noise BEFORE normalization
    data_noisy = add_relative_gaussian_noise(data_clean, nl).astype(np.float32)

    # Compute stats from *noisy training portion*
    X_train_noisy = data_noisy[:, warmup_len:warmup_len + train_len]
    mean = X_train_noisy.mean(axis=1, keepdims=True).astype(np.float32)
    std  = X_train_noisy.std(axis=1, keepdims=True).astype(np.float32)

    # Normalize noisy data with its own stats
    data_norm = ((data_noisy - mean) / std).astype(np.float32)

    # Save
    save_dict = {
        "data": data_norm,   # shape (N, T), float32
        "meta": {
            "warmup_len": warmup_len,
            "train_len": train_len,
            "val_len": val_len,
            "test_len": test_len,
            "mean": mean.squeeze().tolist(),
            "std": std.squeeze().tolist(),
            "noise_scale": nl
        }
    }
    np.save(f"lorenz96_100_noise_{int(nl*100)}.npy", save_dict)
    print(f"Saved lorenz96_100_noise_{int(nl*100)}.npy")


Saved lorenz96_100_noise_1.npy
Saved lorenz96_100_noise_10.npy
Saved lorenz96_100_noise_20.npy
Saved lorenz96_100_noise_30.npy


In [5]:
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------------------------------------
# Load datasets with different noise levels
# -----------------------------------------------------------
noise_levels = [1, 10, 20, 30]
datasets = {}

for nl in noise_levels:
    data = np.load(f"lorenz96_100_noise_{nl}.npy", allow_pickle=True).item()
    datasets[nl] = data["data"].T  # shape (T, 3)

# -----------------------------------------------------------
# Plot: one figure per noise level
# -----------------------------------------------------------
variables = ["x_1", "x_2", "x_3"]
timesteps = np.arange(datasets[0].shape[0])

for nl in noise_levels:
    plt.figure(figsize=(16, 8))
    for i in range(3):  # loop over variables
        plt.subplot(3, 1, i + 1)
        plt.plot(timesteps, datasets[nl][:, i],
                 label=f"{variables[i]} (noise {nl}%)",
                 linewidth=1.0, alpha=0.9)
        plt.title(f"{variables[i]} with {nl}% noise", fontsize=14)
        plt.xlabel("Time step")
        plt.ylabel(variables[i])
        plt.grid(True, linestyle="--", alpha=0.6)
        plt.legend()
        plt.xlim(0, 2000)
    plt.suptitle(f"Lorenz96 100 variables with {nl}% Measurement Noise (first 3)", fontsize=16, y=0.95)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    
    plt.show()


KeyError: 0

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------------------------------------
# Load datasets with different noise levels
# -----------------------------------------------------------
noise_levels = [0, 10, 20, 30]
datasets = {}

for nl in noise_levels:
    data = np.load(f"lorenz96_100_noise_{nl}.npy", allow_pickle=True).item()
    datasets[nl] = data["data"].T  # shape (T, N), transpose → (T, variables)

# -----------------------------------------------------------
# Plot: one figure per noise level
# -----------------------------------------------------------
variables = ["x_1", "x_2", "x_3"]
timesteps = np.arange(datasets[0].shape[0])

for nl in noise_levels:
    plt.figure(figsize=(16, 8))
    for i in range(3):  # loop over variables
        plt.subplot(3, 1, i + 1)
        plt.plot(timesteps, datasets[nl][:, i],
                 label=f"{variables[i]} (noise {nl}%)",
                 linewidth=1.0, alpha=0.9)
        plt.title(f"{variables[i]} with {nl}% noise", fontsize=14)
        plt.xlabel("Time step")
        plt.ylabel(variables[i])
        plt.grid(True, linestyle="--", alpha=0.6)
        plt.legend()
        plt.xlim(0, 2000)

    plt.suptitle(f"Lorenz96 (N=100) – First 3 Variables with {nl}% Noise", fontsize=16, y=0.95)
    plt.tight_layout(rect=[0, 0, 1, 0.96])

    # Save each plot with a unique name
    filename = f"lorenz96_noise_{nl}pct.png"
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    print(f"Saved plot as {filename}")

    plt.show()
